# 09 — 2018 experiment grid, plus the MLP everywhere

Adds two things to `runs.csv`:

1. **A `dataset` column** (schema v2; legacy rows migrate to `2017`).
2. The **2018 grid** — Arm A over all ten days, Arm B on the matched
   Tuesday — and an **MLP** for both datasets under the primary
   (`as_attack`) policy so the cross-dataset table has a neural row.

Compute realities, documented as protocol amendments:
- 2018 Arm A caps training at a stratified 3,000,000 flows per run,
  identically for both releases (comparability across versions is preserved;
  absolute compute is bounded). Test sets are never capped.
- MLP: (64, 32) hidden, adam, batch 8192, max 15 epochs with early stopping —
  one fixed budget applied identically everywhere. Expect this notebook to be
  the longest run of the project; checkpointing makes interruptions free.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np, json

import time, itertools, hashlib, csv, shutil
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, balanced_accuracy_score, matthews_corrcoef,
                             accuracy_score, recall_score, precision_score)
from xgboost import XGBClassifier

RUNS_PATH = os.path.join(C.RESULTS, 'runs.csv')
ALL_COLS = ['run_id','arm','task','dataset','version','split','attempted','model',
            'seed','n_train','n_test','n_features','duplicates_removed',
            'fit_seconds','macro_f1','weighted_f1','balanced_acc','mcc',
            'accuracy','attack_recall','attack_precision','benign_fpr']

def repair_v2():
    lines = [l.rstrip('\n') for l in open(RUNS_PATH) if l.strip()]
    hdr = lines[0].split(',')
    if hdr == ALL_COLS:
        print('schema v2 already —', len(lines)-1, 'rows'); return
    assert 'dataset' not in hdr, 'unexpected header'
    shutil.copy(RUNS_PATH, os.path.join(C.RESULTS, 'runs_backup_v1.csv'))
    recs = []
    for ln in lines[1:]:
        r = dict(zip(hdr, ln.split(',')))
        r['dataset'] = '2017'
        recs.append(r)
    with open(RUNS_PATH, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=ALL_COLS); w.writeheader()
        for r in recs: w.writerow({c: r.get(c, '') for c in ALL_COLS})
    print(f'migrated {len(recs)} rows to schema v2 (dataset=2017)')

repair_v2()

def load_runs():
    return pd.read_csv(RUNS_PATH) if os.path.exists(RUNS_PATH) else pd.DataFrame()

def run_id(**kw):
    return hashlib.md5(json.dumps(kw, sort_keys=True).encode()).hexdigest()[:12]

def append_run(rec):
    new = not os.path.exists(RUNS_PATH)
    with open(RUNS_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=ALL_COLS)
        if new: w.writeheader()
        w.writerow({c: rec.get(c, '') for c in ALL_COLS})

def make_model(name, seed):
    if name == 'logreg':
        return LogisticRegression(max_iter=1000, n_jobs=-1, random_state=seed)
    if name == 'random_forest':
        return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed)
    if name == 'xgboost':
        return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.3,
                             tree_method='hist', n_jobs=-1, random_state=seed,
                             eval_metric='logloss')
    if name == 'mlp':
        return MLPClassifier(hidden_layer_sizes=(64, 32), batch_size=8192,
                             max_iter=15, early_stopping=True,
                             n_iter_no_change=3, random_state=seed)
    raise ValueError(name)

def eval_binary(y, p):
    return {'macro_f1': f1_score(y, p, average='macro', zero_division=0),
            'weighted_f1': f1_score(y, p, average='weighted', zero_division=0),
            'balanced_acc': balanced_accuracy_score(y, p),
            'mcc': matthews_corrcoef(y, p), 'accuracy': accuracy_score(y, p),
            'attack_recall': recall_score(y, p, pos_label=1, zero_division=0),
            'attack_precision': precision_score(y, p, pos_label=1, zero_division=0),
            'benign_fpr': 1.0 - recall_score(y, p, pos_label=0, zero_division=0)}

Mounted at /content/drive
schema v2 already — 360 rows


In [2]:
# Build stratified ANALYSIS SAMPLES for 2018 (amendment, documented):
# the corrected release holds 63.2M flows and the original 16.2M - full-corpus
# training/testing does not fit Colab RAM. Each version is therefore reduced to
# a 4.5M-flow sample, drawn per capture day with stratification on the binary
# label, by ONE fixed procedure applied identically to both versions (master
# seed 7). Day proportions are preserved, so the temporal split stays valid.
# All 2018 Arm A claims are scoped to these samples.
RAW18 = os.path.join(DRIVE_ROOT, 'data/raw_original_2018')
P_O18 = os.path.join(C.INTERIM, 'original_2018_sample.parquet')
P_I18 = os.path.join(C.INTERIM, 'improved_2018_sample.parquet')
TARGET = 4_500_000
MASTER = 7

def shrink(df):
    for c in df.columns:
        if df[c].dtype == 'float64': df[c] = df[c].astype('float32')
        elif df[c].dtype == 'int64': df[c] = df[c].astype('int32')
    return df

def day_sample(df, frac):
    if frac >= 1: return df
    b = H.binarise(df['label'])
    return df.groupby(b, group_keys=False).apply(
        lambda g: g.sample(frac=frac, random_state=MASTER))

def build_sample(list_files, reader, total_rows, out_path, tag):
    if os.path.exists(out_path):
        print(tag, 'sample already cached'); return
    frac = min(1.0, TARGET / total_rows)
    print(f'{tag}: sampling frac={frac:.4f} of {total_rows:,}')
    frames = []
    for fn, path in list_files:
        df = reader(path)
        df['day'] = fn.split('-')[0].lower() + '_' + fn.split('-')[1]
        s = day_sample(df, frac)
        frames.append(shrink(s))
        print(f'  {fn:55s} {len(df):>10,} -> {len(s):>9,}')
        del df
    out = pd.concat(frames, ignore_index=True)
    out.to_parquet(out_path, index=False)
    print(tag, 'sample rows:', len(out)); del frames, out

def read_orig(path):
    df = pd.read_csv(path, low_memory=False, encoding='latin-1')
    df = H.harmonise(df)
    df = df[df['label'].notna() & (df['label'] != 'Label')]
    for c in df.columns:
        if c not in ('label','timestamp','src_ip','dst_ip','flow_id','day'):
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def read_impr(path):
    return H.harmonise(pd.read_csv(path, low_memory=False))

ofiles = [(f, os.path.join(RAW18, f))
          for f in sorted(os.listdir(RAW18)) if f.endswith('.csv')]
ifiles = []
for dirpath, _, files in os.walk(C.RAW_IMPROVED):
    for f in sorted(files):
        if f.lower().endswith('.csv') and '2018' in f:
            ifiles.append((f, os.path.join(dirpath, f)))

O_TOTAL, I_TOTAL = 16_232_943, 63_195_145   # from notebook 07's audit
build_sample(ofiles, read_orig, O_TOTAL, P_O18, 'original')
build_sample(ifiles, read_impr, I_TOTAL, P_I18, 'improved')
print('2018 sample caches ready')

original: sampling frac=0.2772 of 16,232,943
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv          1,048,575 ->   290,680
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv          1,048,574 ->   290,679
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv          1,048,575 ->   290,680
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv        7,948,748 -> 2,203,504
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv          331,100 ->    91,785
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv        1,048,575 ->   290,680
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv        1,048,575 ->   290,679
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv       1,048,575 ->   290,679
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv       1,048,575 ->   290,680
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv         613,071 ->   169,952
original sample rows: 4499998
improved: sampling frac=0.0712 of 63,195,145
  Friday-02-03-2018.csv                                    6

In [3]:
# ---- 2018 grid (binary primary; multiclass random secondary) + MLP ----------
TRAIN_CAP = 3_500_000     # belt-and-braces; samples are already 4.5M
SEEDS = C.SEEDS
# 2018 temporal split: first 6 capture days train, last 4 test (chronological)
def split18(df, proto, seed):
    if proto == 'random_70_30':
        i_tr, i_te = train_test_split(df.index, test_size=0.30, random_state=seed,
                                      stratify=H.binarise(df['label']))
        return df.loc[i_tr], df.loc[i_te]
    days = sorted(df['day'].unique())
    tr_days = days[:max(1, int(len(days) * 0.6))]
    return df[df['day'].isin(tr_days)], df[~df['day'].isin(tr_days)]

def feats_of(tr, te):
    fs = [c for c in tr.columns if c not in
          ('label','label_original','label_improved','attempted','timestamp',
           'src_ip','dst_ip','flow_id','day','version','y')
          and pd.api.types.is_numeric_dtype(tr[c]) and c in te.columns]
    return fs

def cap_train(tr, seed):
    if len(tr) <= TRAIN_CAP: return tr
    _, idx = train_test_split(tr.index, test_size=TRAIN_CAP, random_state=seed,
                              stratify=H.binarise(tr['label']))
    return tr.loc[idx]

def one_run(rec_id, rec, df, proto, model_name, seed, label_col='label'):
    tr, te = split18(df, proto, seed)
    tr = cap_train(tr, seed)
    fs = feats_of(tr, te)
    Xtr = tr[fs].replace([np.inf,-np.inf], np.nan).fillna(0).values
    Xte = te[fs].replace([np.inf,-np.inf], np.nan).fillna(0).values
    ytr = H.binarise(tr[label_col]).values
    yte = H.binarise(te[label_col]).values
    if model_name in ('logreg','mlp'):
        sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    t0 = time.time()
    m = make_model(model_name, seed).fit(Xtr, ytr)
    res = eval_binary(yte, m.predict(Xte))
    append_run({'run_id': rec_id, **rec, 'model': model_name, 'seed': seed,
                'n_train': len(tr), 'n_test': len(te), 'n_features': len(fs),
                'fit_seconds': round(time.time()-t0, 1), **res})
    print(f"{rec['dataset']} {rec['arm']} {rec['version'][:14]:14s} {proto:13s} "
          f"{model_name:13s} s={seed} F1={res['macro_f1']:.4f} "
          f"rec={res['attack_recall']:.4f}")

done = set(load_runs()['run_id'])
MODELS18 = ['logreg', 'random_forest', 'xgboost', 'mlp']

for version, path in [('original', P_O18), ('improved', P_I18)]:
    df = pd.read_parquet(path)
    for proto in ['random_70_30', 'day_ordered']:
        for mdl in MODELS18:
            for seed in SEEDS:
                rid = run_id(arm='A', task='binary', dataset='2018',
                             version=version, split=proto, attempted='as_attack',
                             model=mdl, seed=seed)
                if rid in done: continue
                one_run(rid, {'arm':'A','task':'binary','dataset':'2018',
                              'version':version,'split':proto,
                              'attempted':'as_attack'}, df, proto, mdl, seed)
    del df
print('2018 Arm A binary complete')

2018 A original       random_70_30  logreg        s=11 F1=0.9043 rec=0.8323
2018 A original       random_70_30  logreg        s=23 F1=0.9009 rec=0.8340
2018 A original       random_70_30  logreg        s=37 F1=0.9086 rec=0.8339
2018 A original       random_70_30  logreg        s=51 F1=0.9056 rec=0.8329
2018 A original       random_70_30  logreg        s=73 F1=0.9073 rec=0.8329
2018 A original       random_70_30  random_forest s=11 F1=0.9783 rec=0.9466
2018 A original       random_70_30  random_forest s=23 F1=0.9787 rec=0.9471
2018 A original       random_70_30  random_forest s=37 F1=0.9785 rec=0.9468
2018 A original       random_70_30  random_forest s=51 F1=0.9786 rec=0.9469
2018 A original       random_70_30  random_forest s=73 F1=0.9790 rec=0.9476
2018 A original       random_70_30  xgboost       s=11 F1=0.9817 rec=0.9431
2018 A original       random_70_30  xgboost       s=23 F1=0.9819 rec=0.9438
2018 A original       random_70_30  xgboost       s=37 F1=0.9818 rec=0.9431
2018 A origi

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2018 A original       random_70_30  mlp           s=23 F1=0.9806 rec=0.9407
2018 A original       random_70_30  mlp           s=37 F1=0.9800 rec=0.9395
2018 A original       random_70_30  mlp           s=51 F1=0.9802 rec=0.9392
2018 A original       random_70_30  mlp           s=73 F1=0.9805 rec=0.9408
2018 A original       day_ordered   logreg        s=11 F1=0.6142 rec=0.2632
2018 A original       day_ordered   logreg        s=23 F1=0.6142 rec=0.2632
2018 A original       day_ordered   logreg        s=37 F1=0.6142 rec=0.2632
2018 A original       day_ordered   logreg        s=51 F1=0.6142 rec=0.2632
2018 A original       day_ordered   logreg        s=73 F1=0.6142 rec=0.2632
2018 A original       day_ordered   random_forest s=11 F1=0.6195 rec=0.2508
2018 A original       day_ordered   random_forest s=23 F1=0.6295 rec=0.2653
2018 A original       day_ordered   random_forest s=37 F1=0.6286 rec=0.2639
2018 A original       day_ordered   random_forest s=51 F1=0.5657 rec=0.1782
2018 A origi

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2018 A original       day_ordered   mlp           s=23 F1=0.6600 rec=0.3964
2018 A original       day_ordered   mlp           s=37 F1=0.6690 rec=0.3174
2018 A original       day_ordered   mlp           s=51 F1=0.6690 rec=0.3175
2018 A original       day_ordered   mlp           s=73 F1=0.7139 rec=0.3931
2018 A improved       random_70_30  logreg        s=11 F1=0.9920 rec=0.9828
2018 A improved       random_70_30  logreg        s=23 F1=0.9906 rec=0.9793
2018 A improved       random_70_30  logreg        s=37 F1=0.9916 rec=0.9817
2018 A improved       random_70_30  logreg        s=51 F1=0.9911 rec=0.9808
2018 A improved       random_70_30  logreg        s=73 F1=0.9916 rec=0.9820
2018 A improved       random_70_30  random_forest s=11 F1=0.9999 rec=0.9996
2018 A improved       random_70_30  random_forest s=23 F1=0.9999 rec=0.9996
2018 A improved       random_70_30  random_forest s=37 F1=0.9999 rec=0.9995
2018 A improved       random_70_30  random_forest s=51 F1=0.9999 rec=0.9996
2018 A impro

In [4]:
# ---- 2018 Arm B (matched Tuesday) -------------------------------------------
mt = pd.read_parquet(os.path.join(C.INTERIM, 'matched_2018.parquet'))
mt['day'] = 'tuesday'          # single-day: only random split is meaningful
done = set(load_runs()['run_id'])
for label_col in ['label_original', 'label_improved']:
    for mdl in MODELS18:
        for seed in C.SEEDS:
            rid = run_id(arm='B', task='binary', dataset='2018',
                         labels=label_col, split='random_70_30',
                         model=mdl, seed=seed)
            if rid in done: continue
            df = mt.copy(); df['label'] = df[label_col]
            one_run(rid, {'arm':'B','task':'binary','dataset':'2018',
                          'version':label_col,'split':'random_70_30',
                          'attempted':'n/a'}, df, 'random_70_30', mdl, seed,
                    label_col='label')
print('2018 Arm B complete (random split; single capture day precludes temporal)')

2018 B label_original random_70_30  logreg        s=11 F1=0.9973 rec=0.9918
2018 B label_original random_70_30  logreg        s=23 F1=0.9977 rec=0.9934
2018 B label_original random_70_30  logreg        s=37 F1=0.9973 rec=0.9918
2018 B label_original random_70_30  logreg        s=51 F1=0.9970 rec=0.9901
2018 B label_original random_70_30  logreg        s=73 F1=0.9976 rec=0.9937
2018 B label_original random_70_30  random_forest s=11 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  random_forest s=23 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  random_forest s=37 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  random_forest s=51 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  random_forest s=73 F1=0.9999 rec=0.9997
2018 B label_original random_70_30  xgboost       s=11 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  xgboost       s=23 F1=1.0000 rec=1.0000
2018 B label_original random_70_30  xgboost       s=37 F1=0.9997 rec=0.9997
2018 B label

In [5]:
# ---- MLP top-up for 2017 (as_attack; both splits; both arms) ----------------
done = set(load_runs()['run_id'])
o17 = pd.read_parquet(os.path.join(C.INTERIM, 'original.parquet'))
i17 = pd.read_parquet(os.path.join(C.INTERIM, 'improved.parquet'))
m17 = pd.read_parquet(os.path.join(C.INTERIM, 'matched.parquet'))

def split17(df, proto, seed):
    if proto == 'random_70_30':
        i_tr, i_te = train_test_split(df.index, test_size=0.30, random_state=seed,
                                      stratify=H.binarise(df['label']))
        return df.loc[i_tr], df.loc[i_te]
    return df[df['day'].isin(C.TRAIN_DAYS)], df[df['day'].isin(C.TEST_DAYS)]

global split18
split18_orig = split18
split18 = split17   # reuse one_run against 2017 splits

for version, df in [('original', o17), ('improved', i17)]:
    for proto in ['random_70_30', 'day_ordered']:
        for seed in C.SEEDS:
            rid = run_id(arm='A', task='binary', version=version, split=proto,
                         attempted='as_attack', model='mlp', seed=seed)
            if rid in done: continue
            one_run(rid, {'arm':'A','task':'binary','dataset':'2017',
                          'version':version,'split':proto,'attempted':'as_attack'},
                    df, proto, 'mlp', seed)
for label_col in ['label_original', 'label_improved']:
    for proto in ['random_70_30', 'day_ordered']:
        for seed in C.SEEDS:
            rid = run_id(arm='B', task='binary', labels=label_col, split=proto,
                         model='mlp', seed=seed)
            if rid in done: continue
            df = m17.copy(); df['label'] = df[label_col]
            one_run(rid, {'arm':'B','task':'binary','dataset':'2017',
                          'version':label_col,'split':proto,'attempted':'n/a'},
                    df, proto, 'mlp', seed)
split18 = split18_orig
print('2017 MLP top-up complete')

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2017 A original       random_70_30  mlp           s=11 F1=0.9738 rec=0.9620


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2017 A original       random_70_30  mlp           s=23 F1=0.9756 rec=0.9703


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


...

2017 B label_improved day_ordered   mlp           s=37 F1=0.5112 rec=0.1174


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2017 B label_improved day_ordered   mlp           s=51 F1=0.5111 rec=0.1172


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (15) reached and the optimization hasn't converged yet.
  warnings.warn(


2017 B label_improved day_ordered   mlp           s=73 F1=0.5139 rec=0.1203
2017 MLP top-up complete
